In [1]:
!export HF_HOME="/datastore/clc_hcmus/ZaAIC/hf_cache"

In [2]:
from ultralytics import YOLO

# # Load an official or custom model
# model = YOLO("yolo11n.pt")  # Load an official Detect model

# # Perform tracking with the model
# results = model.track("https://youtu.be/LNwODJXcvt4", show=True, tracker="./trackers/botsort.yaml")  # Tracking with default tracker
# results = model.track("https://youtu.be/LNwODJXcvt4", show=True, tracker="./trackers/bytetrack.yaml")  # with ByteTrack

In [3]:
# import cv2

# from ultralytics import YOLO

# # Load the YOLO11 model
# model = YOLO("yolo11n.pt").to("cuda:7")

# # Open the video file
# video_path = "/datastore/clc_hcmus/ZaAIC/CS412-CV-FinalProject/SUTD/videos/b_1a4411B7sb_clip_005.mp4"
# cap = cv2.VideoCapture(video_path)

# # Loop through the video frames
# while cap.isOpened():
#     # Read a frame from the video
#     success, frame = cap.read()

#     if success:
#         # Run YOLO11 tracking on the frame, persisting tracks between frames
#         results = model.track(frame, persist=True)

#         # Visualize the results on the frame
#         annotated_frame = results[0].plot()

#         # Display the annotated frame
#         cv2.imshow("YOLO11 Tracking", annotated_frame)

#         # Break the loop if 'q' is pressed
#         if cv2.waitKey(1) & 0xFF == ord("q"):
#             break
#     else:
#         # Break the loop if the end of the video is reached
#         break

# # Release the video capture object and close the display window
# cap.release()
# cv2.destroyAllWindows()

In [4]:
# import cv2
# from ultralytics import YOLO
# from IPython.display import display, Image, clear_output
# import matplotlib.pyplot as plt

# # Load the YOLO11 model
# model = YOLO("yolo11n.pt").to("cuda:7")

# # Open the video file
# video_path = "/datastore/clc_hcmus/ZaAIC/CS412-CV-FinalProject/SUTD/videos/b_1a4411B7sb_clip_005.mp4"
# cap = cv2.VideoCapture(video_path)

# # Process just a few frames for display
# max_frames = 5
# frame_count = 0

# fig, axes = plt.subplots(1, min(max_frames, 5), figsize=(20, 4))
# if max_frames == 1:
#     axes = [axes]

# while cap.isOpened() and frame_count < max_frames:
#     success, frame = cap.read()
    
#     if success:
#         # Run YOLO11 tracking on the frame
#         results = model.track(frame, persist=True)
        
#         # Get annotated frame
#         annotated_frame = results[0].plot()
        
#         # Convert BGR to RGB for matplotlib
#         annotated_frame_rgb = cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB)
        
#         # Display in subplot
#         axes[frame_count].imshow(annotated_frame_rgb)
#         axes[frame_count].set_title(f"Frame {frame_count + 1}")
#         axes[frame_count].axis('off')
        
#         frame_count += 1
#     else:
#         break

# cap.release()
# plt.tight_layout()
# plt.show()

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from collections import defaultdict


# Load the YOLO11 model
model = YOLO("yolo11n.pt").to("cuda:7")

# Open the video file
video_path = "/datastore/clc_hcmus/ZaAIC/CS412-CV-FinalProject/SUTD/videos/b_1a4411B7sb_clip_009.mp4"
output_path = "/datastore/clc_hcmus/ZaAIC/CS412-CV-FinalProject/output_tracked_video.mp4"
cap = cv2.VideoCapture(video_path)

# Get video properties for VideoWriter
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Create VideoWriter object
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

# Store the track history
track_history = defaultdict(lambda: [])

# Define colors for different classes (you can customize)
np.random.seed(42)
colors = np.random.randint(0, 255, size=(80, 3), dtype=np.uint8)

frame_count = 0

# Loop through the video frames
while cap.isOpened():
    # Read a frame from the video
    success, frame = cap.read()

    if success:
        # Run YOLO11 tracking on the frame, persisting tracks between frames
        result = model.track(frame, persist=True)[0]

        # Get the boxes and track IDs
        if result.boxes is not None and len(result.boxes) > 0:
            boxes = result.boxes.xyxy.cpu().numpy()  # x1, y1, x2, y2 format
            classes = result.boxes.cls.cpu().numpy().astype(int)
            track_ids = result.boxes.id.int().cpu().tolist() if result.boxes.id is not None else [None] * len(boxes)

            # Draw boxes and class names manually (no ID, no confidence)
            for box, cls, track_id in zip(boxes, classes, track_ids):
                x1, y1, x2, y2 = map(int, box)
                class_name = model.names[cls]
                color = tuple(map(int, colors[cls]))

                # Draw bounding box
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

                # Draw class name only (no ID, no confidence)
                label = class_name
                (label_w, label_h), baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
                cv2.rectangle(frame, (x1, y1 - label_h - 10), (x1 + label_w, y1), color, -1)
                cv2.putText(frame, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

                # Track history for drawing lines
                if track_id is not None:
                    cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
                    track = track_history[track_id]
                    track.append((cx, cy))
                    if len(track) > 30:
                        track.pop(0)

                    # Draw tracking lines
                    points = np.array(track, dtype=np.int32).reshape((-1, 1, 2))
                    cv2.polylines(frame, [points], isClosed=False, color=(230, 230, 230), thickness=2)

        # Write frame to output video
        out.write(frame)
        frame_count += 1
    else:
        # Break the loop if the end of the video is reached
        break

# Release resources
cap.release()
out.release()
print(f"Processed {frame_count} frames. Output saved to: {output_path}")


0: 384x640 1 car, 41.3ms
Speed: 5.8ms preprocess, 41.3ms inference, 35.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 10.4ms
Speed: 3.3ms preprocess, 10.4ms inference, 41.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 12.1ms
Speed: 7.4ms preprocess, 12.1ms inference, 40.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 13.3ms
Speed: 10.1ms preprocess, 13.3ms inference, 39.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 11.7ms
Speed: 6.4ms preprocess, 11.7ms inference, 40.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 11.6ms
Speed: 7.1ms preprocess, 11.6ms inference, 44.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 10.9ms
Speed: 5.9ms preprocess, 10.9ms inference, 38.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 13.8ms
Speed: 8.3ms preprocess, 13.8ms inference, 41.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384

In [9]:
!ffmpeg -y -i "$output_path" -c:v libx264 -preset fast -crf 23 /datastore/clc_hcmus/ZaAIC/CS412-CV-FinalProject/output_tracked_video_h264.mp4

ffmpeg version 7.1 Copyright (c) 2000-2024 the FFmpeg developers
  built with gcc 13.3.0 (conda-forge gcc 13.3.0-1)
  configuration: --prefix=/datastore/clc_hcmus/ZaAIC/envs/cv --cc=/home/conda/feedstock_root/build_artifacts/ffmpeg_1732155166504/_build_env/bin/x86_64-conda-linux-gnu-cc --cxx=/home/conda/feedstock_root/build_artifacts/ffmpeg_1732155166504/_build_env/bin/x86_64-conda-linux-gnu-c++ --nm=/home/conda/feedstock_root/build_artifacts/ffmpeg_1732155166504/_build_env/bin/x86_64-conda-linux-gnu-nm --ar=/home/conda/feedstock_root/build_artifacts/ffmpeg_1732155166504/_build_env/bin/x86_64-conda-linux-gnu-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --disable-gnutls --enable-libmp3lame --enable-libvpx --enable-libass --enable-pthreads --enable-vaapi --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libaom --enable-lib